### Generate latent trajectories


In [ ]:
# ~~~~ Imports ~~~~
import os, random, glob
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde, dirichlet
from scipy.signal import savgol_filter
from sklearn.isotonic import IsotonicRegression
from sklearn.neighbors import NearestNeighbors


# ~~~~ Environment setup ~~~~
device = torch.device('cpu')
print(f"Using device: {device}")

SEED = 29
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

from VAE_model_architectures import *

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

set_seed(SEED)

# ~~~~ Configuration ~~~~ 
class Config:
    BASE_DIR = "/nfs/turbo/umms-kaczoro/u19-shared/vae-HC-PFC/14mon_ADBXD_joined/"
    MODEL_PATH = os.path.join(BASE_DIR, "seed_model/C-GMVAE_SEED_38/saved_model_epoch20000.pth")
    GENE_CSV_PATH = os.path.join(BASE_DIR, "input_datasets/hc_pfc_combined_rp_nor.csv")
    CFM_CSV_PATH  = os.path.join(BASE_DIR, "input_datasets/CFC_CFM_residual_Bins_all_Strains.csv")
    LATENT_SPACE_CSV = os.path.join(BASE_DIR, "seed_model/C-GMVAE_SEED_38/latent_variables_epoch20000.csv")
    RECONS_DATA_CSV  = os.path.join(BASE_DIR, "seed_model/C-GMVAE_SEED_38/recons_cfm.csv")

    GENE_COUNT = 4963
    D_in = D_out = GENE_COUNT
    LATENT_DIM = 10
    H1, H2, H3 = 128, 64, 32

    NUM_INTERPOLATION_STEPS = 24
    DENSITY_WEIGHTS = [0.1, 0.5, 0.9]
    DEFAULT_DENSITY_WEIGHT = 0.5
    NUM_PAIRS = 10
    KNN = 5

In [ ]:
# ~~~~ Configuration ~~~~ 
class Config:
    BASE_DIR = "/home/varnika/draelos_lab/proj_VAE/results/experiments/training20_supcon_knn16_t0.2_b512_ep5k_seed29_normalstart/"
    MODEL_PATH = os.path.join(BASE_DIR, "saved_model_epoch5000.pth")
    GENE_CSV_PATH = "/nfs/turbo/umms-kaczoro/u19-shared/vae-mouse-hc-updated/14_mon_HC_ADBXD_subset/all_cells_adbxd_14mon_rp_nor.csv"
    CFM_CSV_PATH  = "/nfs/turbo/umms-kaczoro/u19-shared/vae-mouse-hc-updated/14_mon_HC_ADBXD_subset/all_cells_adbxd_14mon_rp_nor.csv"
    LATENT_SPACE_CSV = os.path.join(BASE_DIR, "latent_variables_epoch5000.csv")
    RECONS_DATA_CSV  = os.path.join(BASE_DIR, "recons_epoch5000.csv")

    GENE_COUNT = 4842
    D_in = D_out = GENE_COUNT
    LATENT_DIM = 10
    H1, H2, H3 = 128, 64, 32

    NUM_INTERPOLATION_STEPS = 24
    DENSITY_WEIGHTS = [0.1, 0.5, 0.9]
    DEFAULT_DENSITY_WEIGHT = 0.5
    NUM_PAIRS = 10
    KNN = 5

In [ ]:
df = pd.read_csv(Config.GENE_CSV_PATH, index_col = 0)
df

In [ ]:
df.iloc[:, -20:]

In [ ]:
df_recons = pd.read_csv(Config.RECONS_DATA_CSV, index_col = 0)
df_recons

In [ ]:
#df_recons = df_recons.loc[df_recons["epoch"] == 20000] # use only if recons file has all recons across epochs
df_recons

In [ ]:
# ~~~~ Data Loading ~~~~
class MouseGeneDataset(Dataset):
    def __init__(self, df, dim_count=4842, device='cpu'): 
        self.dim_count = dim_count
        self.device = device
        self.x, self.R_S_label, self.cfm = self._process(df)
        self.len = self.x.shape[0]

    def _standardize(self, array):
        return (array - np.mean(array)) / np.std(array)

    def _process(self, df):
        #print(df.columns[-20:])
        x = df.iloc[:, :self.dim_count].values.astype('float32')
        R_S_label = df['14-6-Bin'].values
        cfm = self._standardize(df['14-6-residual'].values)

        return (
            torch.tensor(x).to(self.device),
            torch.tensor(R_S_label).to(self.device),
            torch.tensor(cfm).to(self.device)
        )

    def __getitem__(self, index):
        return self.x[index], self.R_S_label[index], self.cfm[index]

    def __len__(self):
        return self.len


_g = torch.Generator().manual_seed(SEED)

def load_mouse_gene_data(path_gene_csv, path_cfm_csv, dim_count=4842, device='cpu',
                         batch_size=512, shuffle=False, generator=_g):
    df = pd.read_csv(path_gene_csv, index_col=0, dtype={'Strain': str})
    print("Last 20 column names:"); print(df.columns[-20:])
    df = df.sample(frac=1, random_state=42)
    dataset = MouseGeneDataset(df, dim_count=dim_count, device=device)

    def _seed_worker(worker_id):
        worker_seed = SEED + worker_id
        np.random.seed(worker_seed); random.seed(worker_seed); torch.manual_seed(worker_seed)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                        num_workers=0, worker_init_fn=_seed_worker, generator=generator)
    return dataloader, dataset

dataloader, train_set = load_mouse_gene_data(
    path_gene_csv=Config.GENE_CSV_PATH,
    path_cfm_csv=Config.CFM_CSV_PATH,
    dim_count=Config.GENE_COUNT,
    device=device,
    batch_size=512,
    generator=_g)

print("Total samples in dataset:", len(train_set))
print("Shape of one data point:", train_set[0][0].shape)
print("Label of first sample (R/S):", train_set[0][1].item())
print("CFM of first sample:", train_set[0][2].item())

# ~~~~ Model Setup ~~~~

class Autoencoder_CGMVAE(nn.Module):
    #C-GMVAE VAE with GMM prior and conditional layer
    def __init__(self, D_in, D_out, H1=128, H2=64, H3=32, dim_code=10, num_gaussian = 4):
        #Encoder
        super(Autoencoder_CGMVAE, self).__init__()
        self.dim_code = dim_code
        self.num_gaussian = num_gaussian
        self.label = nn.Embedding(num_gaussian, 1)
        self.encoder1 = nn.Sequential(
            nn.Linear(D_in+2, H1),
            nn.BatchNorm1d(H1),
            nn.Sigmoid(),
            nn.Linear(H1,H2),
            nn.BatchNorm1d(H2),
            nn.Sigmoid(),
            nn.Linear(H2,H3),
            nn.BatchNorm1d(H3),
            nn.Sigmoid())
    
        self.mu = nn.Sequential(nn.Linear(H3, out_features = num_gaussian*dim_code))
                            
        self.logsigma = nn.Sequential(nn.Linear(H3, out_features = num_gaussian*dim_code))
        
        self.decoder = nn.Sequential(
            nn.Linear(dim_code+1, H3),
            nn.Sigmoid(),
            nn.BatchNorm1d(H3),
            nn.Linear(H3,H2),
            nn.Sigmoid(),
            nn.BatchNorm1d(H2),
            nn.Linear(H2,H1),
            nn.Sigmoid(),
            nn.BatchNorm1d(H1),
            nn.Linear(H1,D_out+1),
            nn.Sigmoid(),
            nn.BatchNorm1d(D_out+1))
        
    def gaussian_sampler(self, mu, logsigma, y):
        y = y.long().view(-1)
        batch_size = y.shape[0]
        mu_k = mu[torch.arange(batch_size), y]  
        sigma_k = torch.exp(0.5 * logsigma[torch.arange(batch_size), y])
        eps = torch.randn_like(sigma_k)
        z = mu_k + eps * sigma_k
        return mu_k, sigma_k, z
        
    def encode(self, x, y, cfm):
        cfm = cfm.view(cfm.size(0),1)
        if x.dtype != cfm.dtype:
            cfm = cfm.to(x.dtype)
        x = torch.cat((x, cfm), dim=1)
        y_emb = self.label(y)
        y_emb = y_emb.view(y_emb.size(0),1)
        x = torch.cat((x, y_emb), dim=1)
        x = self.encoder1(x)
        mu, logsigma = self.mu(x).view(-1, self.num_gaussian, self.dim_code), self.logsigma(x).view(-1, self.num_gaussian, self.dim_code) 
        mu, logsigma, z = self.gaussian_sampler(mu, logsigma, y)
        return mu, logsigma, z
    
    def decode(self, x, y):
        y_emb = self.label(y)
        y_emb = y_emb.view(y_emb.size(0),1)
        x = torch.cat((x, y_emb), dim=1)
        reconstruction = self.decoder(x)
        return reconstruction
    
    def forward(self, x, y, cfm):
        cfm = cfm.view(cfm.size(0),1)
        if x.dtype != cfm.dtype:
            cfm = cfm.to(x.dtype)
        x = torch.cat((x, cfm), dim=1)
        y_emb = self.label(y)
        y_emb = y_emb.view(y_emb.size(0),1)
        x = torch.cat((x, y_emb), dim=1)
        x = self.encoder1(x)
        mu, logsigma = self.mu(x).view(-1, self.num_gaussian, self.dim_code), self.logsigma(x).view(-1, self.num_gaussian, self.dim_code) 
        mu, logsigma, z = self.gaussian_sampler(mu, logsigma,y)
        u = torch.cat((z, y_emb), dim=1)
        reconstruction = self.decoder(u)
        return mu, logsigma, z, reconstruction

class ProjectionHead(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int = 16,
        out_dim: int = 32,
        dropout: float = 0.0,
        use_batchnorm: bool = False,
        normalize: bool = False,
    ):
        super().__init__()
        self.normalize = normalize

        layers = [nn.Linear(in_dim, hidden_dim)]

        if use_batchnorm:
            layers.append(nn.BatchNorm1d(hidden_dim))

        layers.append(nn.ReLU(inplace=True))

        if dropout > 0:
            layers.append(nn.Dropout(dropout))

        layers.append(nn.Linear(hidden_dim, out_dim))

        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        p = self.net(x)
        if self.normalize:
            p = F.normalize(p, p=2, dim=1)
        return p
        
D_in = D_out = 4842
H1, H2, H3, latent_dim = 128, 64, 32, 10

model = Autoencoder_CGMVAE(D_in, D_out, H1, H2, H3, latent_dim).to(device)
model = torch.load(Config.MODEL_PATH, map_location=device, weights_only = False)
model.to(device)
print("Model loaded.")

In [ ]:
 ~~~~ Latent Space Data ~~~~
df_latent = pd.read_csv(Config.LATENT_SPACE_CSV, index_col=0)
print(df_latent['14-6-Bin'].value_counts())
print(df_latent.describe())
print(df_latent)
cols = list(df_latent.columns[:Config.LATENT_DIM]) + ["14-6-Bin"]
df_latent = df_latent[cols].copy()
df_latent.columns = [f"z_{i+1}" for i in range(Config.LATENT_DIM)] + ["label"]

df_recons = pd.read_csv(Config.RECONS_DATA_CSV)
#df_recons = df_recons.loc[df_recons["epoch"] == 20000] # use only if recons file has all recons across epochs
df_latent['decoded_cfm'] = df_recons['CFM'].values # check column name of cfm - 'CFM' or 'cfm'


def get_latent_interpolation_endpoints(df_latent, latent_dim=10, num_pairs=1, seed=SEED):
    # --- set seed for reproducibility ---
    random.seed(seed)
    np.random.seed(seed)
    
    pairs = []
    df_sorted = df_latent.sort_values(by='decoded_cfm', ignore_index=True)
    low_candidates  = df_sorted.head(100)
    high_candidates = df_sorted.tail(100)
    
    for i in range(num_pairs):
        rng = np.random.default_rng(seed + i)
        low_sample  = low_candidates.sample(1, random_state=seed + i).iloc[0]
        high_sample = high_candidates.sample(1, random_state=seed + 1000 + i).iloc[0]
        tries = 0
        while low_sample['label'] == high_sample['label'] and tries < 10:
            high_sample = high_candidates.sample(1, random_state=seed + 2000 + i + tries).iloc[0]
            tries += 1
        low_dict = {'z': low_sample[[f'z_{j+1}' for j in range(latent_dim)]].values.astype('float32'),
            'label': int(low_sample['label']),
            'cfm': float(low_sample['decoded_cfm'])}
        high_dict = {'z': high_sample[[f'z_{j+1}' for j in range(latent_dim)]].values.astype('float32'),
            'label': int(high_sample['label']),
            'cfm': float(high_sample['decoded_cfm'])}
        pairs.append((low_dict, high_dict))
    
    return pairs
    
pairs = get_latent_interpolation_endpoints(df_latent, latent_dim=Config.LATENT_DIM, num_pairs=Config.NUM_PAIRS, seed=SEED)

for i, (low, high) in enumerate(pairs):
    print(f"Pair {i+1}:\n  Low -> label={low['label']}, CFM={low['cfm']:.4f}\n  High-> label={high['label']}, CFM={high['cfm']:.4f}\n")

# ~~~~ Density-Guided Interpolation ~~~~
def log_density(z_point, kde, eps=1e-9):
    """
    Compute log-density at a point using KDE.
    
    Args:
        z_point (np.ndarray): A single latent point (shape: [latent_dim,] or [latent_dim, 1]).
        kde (gaussian_kde): Fitted KDE object on the latent space.
        eps (float): Small value to avoid log(0).

    Returns:
        float: log-density at the point.
    """
    z_point = z_point.reshape(-1, 1)  # Ensure correct shape for KDE
    density_val = kde(z_point)[0]
    return np.log(density_val + eps)


def numerical_log_density_gradient(z_point, kde, epsilon=1e-4):
    """
    Estimate the gradient of log-density at a point using central difference.

    Args:
        z_point (np.ndarray): Point in latent space.
        kde (gaussian_kde): KDE fitted on latent space.
        epsilon (float): Small perturbation for numerical gradient.

    Returns:
        np.ndarray: Estimated gradient vector of same shape as z_point.
    """
    grad = np.zeros_like(z_point)
    for j in range(len(z_point)):
        perturb = np.zeros_like(z_point)
        perturb[j] = epsilon

        ld_plus = log_density(z_point + perturb, kde)
        ld_minus = log_density(z_point - perturb, kde)
        grad[j] = (ld_plus - ld_minus) / (2.0 * epsilon)

    return grad


def density_guided_interpolation(
    start_point,
    end_point,
    latent_space,
    steps,
    step_size,
    density_weight=0.8,
    log_density_drop_threshold=0.5,
    min_step_size=1e-2
):
    """
    Generate an interpolation path in latent space guided by KDE density.

    Args:
        start_point (np.ndarray): Starting latent vector.
        end_point (np.ndarray): Target latent vector.
        latent_space (np.ndarray): All latent vectors from data (for KDE fitting).
        steps (int): Number of interpolation steps.
        step_size (float): Base step size.
        density_weight (float): Weight of density gradient vs. target direction.
        log_density_drop_threshold (float): Threshold for reducing step size if moving into low-density region.
        min_step_size (float): Minimum allowed step size.

    Returns:
        np.ndarray: Interpolated latent path (shape: [steps + 2, latent_dim]).
    """
    kde = gaussian_kde(latent_space.T)  # Fit KDE on training latent space

    path = [start_point]
    current_point = start_point.copy()

    for _ in range(steps):
        cur_step_size = step_size

        # Direction toward the goal
        target_dir = end_point - current_point
        target_norm = np.linalg.norm(target_dir)
        if target_norm > 0:
            target_dir /= target_norm

        # Direction toward denser region
        density_grad = numerical_log_density_gradient(current_point, kde)
        grad_norm = np.linalg.norm(density_grad)
        if grad_norm > 0:
            density_grad /= grad_norm

        # Weighted blend of directions
        move_dir = (1.0 - density_weight) * target_dir + density_weight * density_grad
        move_norm = np.linalg.norm(move_dir)
        if move_norm > 0:
            move_dir /= move_norm

        # Propose next step
        proposed_next_point = current_point + move_dir * cur_step_size

        # Adjust step size if moving into much lower-density region
        current_log_dens = log_density(current_point, kde)
        next_log_dens = log_density(proposed_next_point, kde)

        if next_log_dens < current_log_dens - log_density_drop_threshold:
            cur_step_size = max(cur_step_size * 0.9, min_step_size)

        # Finalize step
        next_point = current_point + move_dir * cur_step_size
        path.append(next_point)
        current_point = next_point

    # Add final target point explicitly
    path.append(end_point)
    return np.array(path)


def compute_interpolations_between_points(
    start_point,
    end_point,
    latent_space,
    num_steps=24,
    density_weights=[0.1, 0.5, 0.9]
):
    """
    Compute multiple interpolation paths using different density weights.

    Args:
        start_point (np.ndarray): Start latent vector.
        end_point (np.ndarray): End latent vector.
        latent_space (np.ndarray): Full latent dataset for KDE.
        num_steps (int): Number of steps per interpolation.
        density_weights (list): Density weights to try (0 = straight line, 1 = fully density-driven).

    Returns:
        dict: {density_weight: interpolated_path} mappings.
    """
    total_distance = np.linalg.norm(end_point - start_point)
    if total_distance < 1e-6:
        return {dw: np.tile(start_point, (num_steps + 2, 1)) for dw in density_weights}

    step_size = total_distance / num_steps

    interpolations = {}
    for dw in density_weights:
        interpolated_path = density_guided_interpolation(
            start_point=start_point,
            end_point=end_point,
            latent_space=latent_space,
            steps=num_steps,
            step_size=step_size,
            density_weight=dw
        )
        interpolations[dw] = interpolated_path

    return interpolations

In [ ]:
# ~~~~ Label Assignment ~~~~
def get_local_label_counts(z_point, latent_space, labels, k=20):
    """
    For a single latent point, find the k nearest neighbors and count their labels.
    """
    nbrs = NearestNeighbors(n_neighbors=k).fit(latent_space)
    _, indices = nbrs.kneighbors(z_point.reshape(1, -1))
    neighbor_labels = labels[indices[0]]
    unique_labels, counts = np.unique(neighbor_labels, return_counts=True)
    return {label: count for label, count in zip(unique_labels, counts)}

def get_expected_label_dirichlet(label_counts, all_labels, alpha_prior=1.0):
    """
    Compute the expected class from a Dirichlet distribution over local label frequencies.
    """
    alpha_vec = np.array([label_counts.get(lbl, 0) + alpha_prior for lbl in all_labels])
    dirichlet_probs = alpha_vec / np.sum(alpha_vec) # Expected value of Dirichlet
    expected_class = all_labels[np.argmax(dirichlet_probs)]
    return expected_class, dirichlet_probs

def assign_labels_dirichlet_expected(interpolated_path, latent_space, labels, k=20, alpha_prior=1.0):
    """
    Assign labels to a trajectory by computing the expected class under local Dirichlet distributions.

    Returns:
        assigned_labels: np.ndarray of assigned labels
        all_probs: np.ndarray of Dirichlet expected probabilities (shape: [num_points, num_classes])
        uncertainties: np.ndarray of entropy (per-point uncertainty)
    """
    all_labels = np.unique(labels)
    assigned_labels = []
    all_probs = []
    uncertainties = []

    for z in interpolated_path:
        label_counts = get_local_label_counts(z, latent_space, labels, k)
        expected_class, dirichlet_probs = get_expected_label_dirichlet(label_counts, all_labels, alpha_prior)
        
        assigned_labels.append(expected_class)
        all_probs.append(dirichlet_probs)
        
        entropy = -np.sum(dirichlet_probs * np.log(dirichlet_probs + 1e-8))
        uncertainties.append(entropy)

    return np.array(assigned_labels), np.vstack(all_probs), np.array(uncertainties)

def decode_cfm_along_trajectory(model, trajectory_z, trajectory_labels, device='cpu'):
    """
    Decode the latent trajectory to get reconstructed CFM values.

    Args:
        model: The full model with .decode(z, y)
        trajectory_z (np.ndarray): Array of shape [steps, latent_dim]
        trajectory_labels (np.ndarray): Array of shape [steps] or [steps, label_dim]
        device (str): Device to run model on

    Returns:
        np.ndarray: Reconstructed CFM values (shape: [steps])
    """
    model.eval()

    z_tensor = torch.tensor(trajectory_z, dtype=torch.float32).to(device)

    # If labels are not already one-hot or embedded, just use as is
    if isinstance(trajectory_labels[0], (np.integer, int)):
        y_tensor = torch.tensor(trajectory_labels, dtype=torch.long).to(device)
    else:
        y_tensor = torch.tensor(trajectory_labels, dtype=torch.float32).to(device)

    with torch.no_grad():
        decoded = model.decode(z_tensor, y_tensor) # shape: [steps, output_dim]
        cfm_values = decoded[:, -1].cpu().numpy() # last column = predicted CFM

    return cfm_values

all_interpolations = {}
latent_space = df_latent[[f'z_{j+1}' for j in range(Config.LATENT_DIM)]].values.astype('float32')
decoded_cfm_all = df_latent["decoded_cfm"].values 

# ONLY edit this and the respective out_dir in save_full_trajectory_and_latents()
# Change start_point=end_point and end_point=start_point if you want to do reverse trajectories
for i, (low, high) in enumerate(pairs):
    start_point, end_point = low['z'], high['z']
    interpolations = compute_interpolations_between_points(
        start_point=start_point,
        end_point=end_point,
        latent_space=latent_space,
        num_steps=Config.NUM_INTERPOLATION_STEPS,
        density_weights=Config.DENSITY_WEIGHTS
    )

    label_assignments = {}
    for dw, path in interpolations.items():
        assigned_labels, label_probs, uncertainties = assign_labels_dirichlet_expected(
            interpolated_path=path,
            latent_space=latent_space,
            labels=df_latent['label'].values,
            k=Config.KNN, alpha_prior=1.0
        )
        label_assignments[dw] = {'labels': assigned_labels, 'probs': label_probs, 'uncertainty': uncertainties}
        cfm_values = decode_cfm_along_trajectory(model, path, assigned_labels, device=device)
        label_assignments[dw]['decoded_cfm'] = cfm_values

    all_interpolations[f"pair_{i+1}"] = {'paths': interpolations, 'labels': label_assignments}

# ~~~~ Plot ~~~~
def plot_decoded_cfm(cfm_dict, pair_idx, cfm_csv_path=Config.CFM_CSV_PATH, title="", figsize=(10, 7), smooth=True, window=5, poly=2):
    """
    Plot decoded CFM across all 26 steps along thd ttrajectories.
    window = 5: number of steps for local averaging
    poly = 2: quadratic fit (keeps curvature natural)
    smooth = False: for raw curves.
    """
    plt.figure(figsize=figsize)
    df_cfm = pd.read_csv(cfm_csv_path)
    mean_cfm = df_cfm['CFM_14_5_snRNA'].mean()
    std_cfm  = df_cfm['CFM_14_5_snRNA'].std()
    viridis_colors = plt.cm.viridis(np.linspace(0, 1, len(cfm_dict)))
    for i, (dw, cfm_values) in enumerate(sorted(cfm_dict.items())):
        cfm_values = np.array(cfm_values, dtype=float)
        if smooth and len(cfm_values) >= window:
            smoothed = savgol_filter(cfm_values, window_length=window, polyorder=poly)
            smoothed[0] = cfm_values[0]
            smoothed[-1] = cfm_values[-1]
            cfm_values = smoothed
        cfm_values = cfm_values * std_cfm + mean_cfm
        plt.plot(range(len(cfm_values)), cfm_values,
                 label=f'Density weight {dw}',
                 color=viridis_colors[i], marker='o',
                 linewidth=3.2, markersize=7, alpha=0.9)
    plt.xlabel("Step in trajectory", fontsize=20)
    plt.ylabel("14 month CFM score", fontsize=20)
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)
    plt.title((title), fontsize=12)
    plt.grid(True, linestyle='-', linewidth=1.0, color='gray', alpha=0.8) 
    plt.legend(fontsize=15, frameon=True, fancybox=True, framealpha=0.9, borderpad=1.2, loc='best')
    plt.tight_layout()
    #fig_path_jpg = f"/home/varnika/draelos_lab/proj_VAE/results/miscellaneous/YC_HC_PFC/pair{pair_idx+1}_trajectories_reverse.jpg"
    #plt.savefig(fig_path_jpg, dpi=300, bbox_inches="tight")
    plt.show()


for i, (low, high) in enumerate(pairs):
    cfm_dict = {dw: all_interpolations[f"pair_{i+1}"]['labels'][dw]['decoded_cfm']
                for dw in all_interpolations[f"pair_{i+1}"]['labels']}
    plot_decoded_cfm(cfm_dict, i, cfm_csv_path=Config.CFM_CSV_PATH, title="", smooth=False, window=19)

In [ ]:
def save_full_trajectory_and_latents(all_interpolations, model, device,
                                     out_dir="/home/varnika/draelos_lab/proj_VAE/results/experiments/training20_supcon_knn16_t0.2_b512_ep5k_seed29_normalstart/reconstructed_pairs"):
    """
    For each (pair_id, density_weight):
      1. Saves *_recon.csv - reconstructed genes + decoded_cfm
      2. Saves *_latent.csv - latent z's + label + uncertainty + decoded_cfm

    Automatically decodes missing reconstructions when needed.
    """
    os.makedirs(out_dir, exist_ok=True)

    for pair_id, data in all_interpolations.items():
        paths_dict = data["paths"]
        labels_dict = data["labels"]

        print(f"Processing {pair_id} | densities: {list(paths_dict.keys())}")

        for dw, path in paths_dict.items():
            if dw not in labels_dict:
                print(f"Skipping {pair_id} density {dw}: no labels found.")
                continue

            label_data = labels_dict[dw]
            labels = np.asarray(label_data["labels"])
            probs = np.asarray(label_data["probs"])
            uncertainty = np.asarray(label_data["uncertainty"])

            # Decode if missing 
            if "reconstructed_genes" not in label_data:
                z_tensor = torch.tensor(path, dtype=torch.float32).to(device)
                y_tensor = torch.tensor(labels, dtype=torch.long).to(device)

                with torch.no_grad():
                    decoded = model.decode(z_tensor, y_tensor).cpu().numpy()  # [steps, genes+1]
                    cfm_values = decoded[:, -1] # last column is CFM
                    gene_recon = decoded[:, :-1] # all others are genes

                label_data["reconstructed_genes"] = gene_recon
            elif "decoded_cfm" not in label_data and "reconstructed_genes" not in label_data:
                z_tensor = torch.tensor(path, dtype=torch.float32).to(device)
                y_tensor = torch.tensor(labels, dtype=torch.long).to(device)

                with torch.no_grad():
                    decoded = model.decode(z_tensor, y_tensor).cpu().numpy()  # [steps, genes+1]
                    cfm_values = decoded[:, -1] # last column is CFM
                    gene_recon = decoded[:, :-1] # all others are genes

                label_data["decoded_cfm"] = cfm_values
                label_data["reconstructed_genes"] = gene_recon
            else:
                cfm_values = np.asarray(label_data["decoded_cfm"])
                gene_recon = np.asarray(label_data["reconstructed_genes"])

            n_steps, latent_dim = path.shape

            df_base = pd.DataFrame({
                "pair_id": [pair_id] * n_steps,
                "density_weight": [dw] * n_steps,
                "step": np.arange(n_steps),
                "label": labels,
                "label_prob": np.max(probs, axis=1),
                "uncertainty": uncertainty,
                "decoded_cfm": cfm_values
            })

            gene_cols = [f"gene_{j}" for j in range(gene_recon.shape[1])]
            df_genes = pd.DataFrame(gene_recon, columns=gene_cols)
            df_recon = pd.concat([df_base, df_genes], axis=1)

            recon_path = os.path.join(out_dir, f"{pair_id}_dw{dw}_recon.csv") 
            df_recon.to_csv(recon_path, index=False)
            print(f"Saved {recon_path}")

            df_latent = pd.DataFrame({
                "pair_id": [pair_id] * n_steps,
                "density_weight": [dw] * n_steps,
                "step": np.arange(n_steps),
                "label": labels,
                "label_prob": np.max(probs, axis=1),
                "uncertainty": uncertainty,
                "decoded_cfm": cfm_values
            })

            latent_cols = [f"z{j}" for j in range(latent_dim)]
            df_latent = pd.concat([df_base, pd.DataFrame(path, columns=latent_cols)], axis=1)

            latent_path = os.path.join(out_dir, f"{pair_id}_dw{dw}_latent.csv")
            df_latent.to_csv(latent_path, index=False)
            print(f"Saved {latent_path}")

    print(f"\nAll trajectory files saved under: {out_dir}")

save_full_trajectory_and_latents(all_interpolations, model, device)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap
from scipy.signal import savgol_filter

density = 0.5
df_cfm = pd.read_csv("/nfs/turbo/umms-kaczoro/u19-shared/vae-HC-PFC/14mon_ADBXD_joined/input_datasets/CFC_CFM_residual_Bins_all_Strains.csv")
mean_cfm = df_cfm['CFM_14_5_snRNA'].mean()
std_cfm  = df_cfm['CFM_14_5_snRNA'].std()
forward_dir = "/home/varnika/draelos_lab/proj_VAE/results/miscellaneous/YC_HC_PFC/reconstructed_pairs"
reverse_dir = "/home/varnika/draelos_lab/proj_VAE/results/miscellaneous/YC_HC_PFC/reconstructed_pairs_reverse"
pair_ids = [f"pair_{i+1}" for i in range(10)] 
smooth = True
window = 8
poly = 2

plt.figure(figsize=(8, 7), dpi=300)

for idx, pair_id in enumerate(pair_ids):
    fwd_path = os.path.join(forward_dir, f"{pair_id}_dw{density}_latent.csv")
    rev_path = os.path.join(reverse_dir, f"{pair_id}_dw{density}_latent.csv")

    if os.path.exists(fwd_path):
        df_fwd = pd.read_csv(fwd_path)
        cfm_fwd = df_fwd["decoded_cfm"] 
        if smooth and len(cfm_fwd) >= window:
            cfm_fwd = savgol_filter(cfm_fwd, window_length=window, polyorder=poly)
            cfm_fwd = cfm_fwd * std_cfm + mean_cfm 
        plt.plot(
            range(len(cfm_fwd)),
            cfm_fwd,
            label=f"{pair_id} (fwd)",
            color="grey",  # solid
            linestyle='-',
            marker='o',
            markersize=3
        )

    if os.path.exists(rev_path):
        df_rev = pd.read_csv(rev_path)
        cfm_rev = df_rev["decoded_cfm"]
        if smooth and len(cfm_rev) >= window:
            cfm_rev = savgol_filter(cfm_rev, window_length=window, polyorder=poly)
            cfm_rev = cfm_rev * std_cfm + mean_cfm
        plt.plot(
            range(len(cfm_rev)),
            cfm_rev,
            label=f"{pair_id} (rev)",
            color="red", # dotted
            linestyle='--',
            marker='o',
            markersize=3
        )

# Remove only top and right spines (keep bottom and left for axes)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
    
plt.title("")
plt.xlabel("Step along trajectory")
plt.ylabel("CFM")
plt.ylim(-50,600)
plt.grid(False)
plt.tight_layout()
plt.savefig("/home/varnika/draelos_lab/proj_VAE/results/miscellaneous/YC_HC_PFC/fig7a_2.jpg", dpi=300)

### Functional enrichment analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import pickle
from scipy.stats import pearsonr
from collections import Counter
import seaborn as sns
from scipy.stats import skew
from gprofiler import GProfiler
import mygene
from collections import defaultdict
import gseapy as gp
from scipy.stats import spearmanr
from tqdm import tqdm

In [ ]:
# Configs
# ONLY edit TRAJ_DIR for reconstructed_pairs_reverse or reconstructed_pairs, and save to same OUTPUT_DIR. 
# When evaluating reverse trajectories, save thier names as so save_path = os.path.join(OUTPUT_DIR, f"pair{i+10}_dw0.5_recon_orig.csv")
# This way, in OUTPUT_DIR, pairs 1 to 10 - forward, pairs 11 to 20 are reverse 
# Change max_eff_corr as threshold as necessary
BASE_DIR = "/nfs/turbo/umms-kaczoro/u19-shared/vae-mouse-hc-updated/14_mon_HC_ADBXD_subset"
PIPELINE_PATH = os.path.join(BASE_DIR, "random_projection.pkl")
GENE_LIST_PATH = os.path.join(BASE_DIR, "gene_list.csv")
TRAJ_DIR = "/home/varnika/draelos_lab/proj_VAE/results/baseline/CGMVAE_30percent/reconstructed_pairs_reverse"
OUTPUT_DIR = "/home/varnika/draelos_lab/proj_VAE/results/baseline/CGMVAE_30percent/reconstructed_pairs_hd"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def rp_to_rawCount(pipeline, df_rp):
    """Inverse transform from RP space to reconstructed raw count space."""
    rp = pipeline.named_steps["random_projection"] # GaussianRandomProjection
    scaler = pipeline.named_steps["scaler"] # StandardScaler

    Y_scaled = df_rp.values
    Y = scaler.inverse_transform(Y_scaled) # Shape: (n_samples, n_components)

    A = rp.components_ # Shape: (n_components, n_features)
    X_hat = Y @ np.linalg.pinv(A.T) # Shape: (n_samples, n_features)
    return X_hat

pipeline = joblib.load(PIPELINE_PATH)
gene_list = pd.read_csv(GENE_LIST_PATH)

# Mapping gene_orig_i to real gene name
gene_ids = [f"gene_orig_{i+1}" for i in range(len(gene_list))]
gene_names = gene_list["gene"].values
gene_mapping = pd.DataFrame({"gene_id": gene_ids, "gene_name": gene_names})
gene_mapping.to_csv(os.path.join(OUTPUT_DIR, "gene_id_to_real_name_mapping.csv"), index=False)
print(f"Saved mapping: {os.path.join(OUTPUT_DIR, 'gene_id_to_real_name_mapping.csv')}")

# Process 10 trajectories (forward/reverse)
for i in range(1, 11):
    traj_path = os.path.join(TRAJ_DIR, f"pair_{i}_dw0.5_recon.csv")
    if not os.path.exists(traj_path):
        print(f"File not found: {traj_path}")
        continue

    print(f"Processing {traj_path}:")
    df_traj = pd.read_csv(traj_path)

    # Extract latent gene columns
    gene_cols = [c for c in df_traj.columns if c.startswith("gene_")]
    df_latent = df_traj[gene_cols]

    # Reconstruct to original gene space
    X_hat = rp_to_rawCount(pipeline, df_latent)
    df_recon = pd.DataFrame(X_hat, columns=gene_mapping["gene_id"], index=df_traj.index)

    # Add decoded_cfm and label
    if "decoded_cfm" in df_traj.columns:
        df_recon["decoded_cfm"] = df_traj["decoded_cfm"]
    if "label" in df_traj.columns:
        df_recon["label"] = df_traj["label"]

    # Save output
    save_path = os.path.join(OUTPUT_DIR, f"pair{i+10}_dw0.5_recon_orig.csv")
    df_recon.to_csv(save_path, index=False)
    print(f"Saved reconstructed trajectory: {save_path}")

print("\nReconstructed trajectories saved in:")
print(f"{OUTPUT_DIR}")

In [ ]:
base_dir = "/home/varnika/draelos_lab/proj_VAE/results/baseline/CGMVAE_30percent/"
data_dir = os.path.join(base_dir, "reconstructed_pairs_hd")
# NAME DEPENDS ON max_eff_corr_threshold
gene_list_dir = os.path.join(base_dir, "Gene_lists_0.65")
os.makedirs(gene_list_dir, exist_ok=True)

rp = joblib.load("/nfs/turbo/umms-kaczoro/u19-shared/vae-mouse-hc-updated/14_mon_HC_ADBXD_subset/random_projection.pkl").named_steps["random_projection"]
W = rp.components_
mapping_df = pd.read_csv(os.path.join(data_dir, "gene_id_to_real_name_mapping.csv"))
mapping_dict = dict(zip(mapping_df.iloc[:, 0], mapping_df.iloc[:, 1]))

recon_dir_fwd = os.path.join(base_dir, "reconstructed_pairs")
recon_dir_rev = os.path.join(base_dir, "reconstructed_pairs_reverse")

for i in range(1, 21):
    print(f"Processing pair {i}")

    original_path = os.path.join(data_dir, f"pair{i}_dw0.5_recon_orig.csv")

    if i <= 10:
        reduced_path = os.path.join(recon_dir_fwd, f"pair_{i}_dw0.5_recon.csv")
    else:
        reduced_path = os.path.join(recon_dir_rev, f"pair_{i-10}_dw0.5_recon.csv")

    original_df = pd.read_csv(original_path)
    reduced_df = pd.read_csv(reduced_path)
    original_data = original_df.iloc[:, :-2]
    original_cfm = original_df.iloc[:, -2]
    reduced_data = reduced_df[[c for c in reduced_df.columns if c.startswith("gene_")]]
    reduced_cfm = reduced_df["decoded_cfm"]

    reduced_corr = reduced_data.apply(lambda col: pearsonr(col, reduced_cfm)[0])
    original_corr = original_data.apply(lambda col: pearsonr(col, original_cfm)[0])

    W_df = pd.DataFrame(W, index=reduced_data.columns, columns=original_data.columns)

    percentile_cutoff = 99.9
    results, max_genes = [], 0

    for j, row in W_df.iterrows():
        abs_row = row.abs()
        threshold = np.percentile(abs_row, percentile_cutoff)
        top_genes = abs_row[abs_row >= threshold].sort_values(ascending=False)
        result_row = {"latent_gene_feature": j, "pearson_corr_with_cfm": reduced_corr[j]}
        for k, gene_idx in enumerate(top_genes.index):
            result_row[f"top_{k+1}_gene"] = gene_idx
            result_row[f"top_{k+1}_gene_weight"] = row[gene_idx]
        max_genes = max(max_genes, len(top_genes))
        results.append(result_row)

    for row in results:
        for k in range(1, max_genes + 1):
            row.setdefault(f"top_{k}_gene", np.nan)
            row.setdefault(f"top_{k}_gene_weight", np.nan)

    results_df = pd.DataFrame(results)
    print(results_df.shape)

    gene_latent_assoc = defaultdict(list)
    gene_max_eff_corr = defaultdict(float)
    gene_occurrences = defaultdict(int)
    gene_max_weight = defaultdict(float)

    for _, row in results_df.iterrows():
        latent = row['latent_gene_feature']
        cfm_corr = row['pearson_corr_with_cfm']
        for k in range(1, 33):
            gene_col = f"top_{k}_gene"
            weight_col = f"top_{k}_gene_weight"
            gene = row[gene_col]
            weight = row[weight_col]

            if pd.isna(gene) or gene not in W_df.columns:
                continue

            eff_corr = cfm_corr * np.sign(weight)
            gene_latent_assoc[gene].append((latent, weight, cfm_corr))
            gene_occurrences[gene] += 1

            if abs(eff_corr) > abs(gene_max_eff_corr[gene]):
                gene_max_eff_corr[gene] = eff_corr

            if abs(weight) > abs(gene_max_weight[gene]):
                gene_max_weight[gene] = weight

    C1, C2, C3, C4 = set(), set(), set(), set()
    # EDIT THIS max_eff_corr threshold as necessary 
    for gene, max_eff_corr in gene_max_eff_corr.items():
        if max_eff_corr >= 0.65: 
            C1.add(gene)
            #C3.add(gene)
        elif max_eff_corr <= -0.65:
            C2.add(gene)
            #C4.add(gene)

    def to_df(gene_set, score_dict=None, score_name="Score"):
        df = pd.DataFrame({"gene_orig_id": list(gene_set)})
        df["real_gene_name"] = df["gene_orig_id"].map(mapping_dict)
        if score_dict:
            df[score_name] = df["gene_orig_id"].map(score_dict)
            df = df.sort_values(score_name, ascending=False)
        return df.dropna(subset=["real_gene_name"]).drop_duplicates("real_gene_name")

    df_C1 = to_df(C1, gene_occurrences, "Occurrences")
    df_C2 = to_df(C2, gene_occurrences, "Occurrences")
    #df_C3 = to_df(C3, gene_max_weight, "MaxWeight")
    #df_C4 = to_df(C4, gene_max_weight, "MaxWeight")

    gp = GProfiler(return_dataframe=True)
    for label, df in zip(["C1", "C2"], [df_C1, df_C2]):
        query_names = df["real_gene_name"].dropna().unique().tolist()
        mapping = gp.convert(organism="mmusculus", query=query_names)
        if not isinstance(mapping, pd.DataFrame):
            # Fallback for new API format
            mapping = pd.DataFrame(mapping)
        # handle both old and new schema names
        if 'name' not in mapping.columns and 'target' in mapping.columns:
            mapping.rename(columns={'target': 'name', 'input': 'incoming'}, inplace=True)
        mapping_clean = mapping.drop_duplicates(subset='incoming', keep='first')
        mask_predicted = mapping_clean['name'].str.startswith(('Gm', 'LOC'), na=False)
        mask_riken = mapping_clean['name'].str.contains('Rik$', na=False)
        mask_nan = mapping_clean['name'].isna()
        filtered = mapping_clean[~(mask_predicted | mask_riken | mask_nan)]

        mg = mygene.MyGeneInfo()
        gene_list = filtered['name'].dropna().unique().tolist()
        gene_info = mg.querymany(gene_list, scopes='symbol', fields='type_of_gene', species='mouse')
        gene_info_df = pd.DataFrame(gene_info)
        merged = pd.merge(filtered, gene_info_df, left_on='name', right_on='query', how='inner')
        protein_coding = merged[merged['type_of_gene'] == 'protein-coding']
        valid_real_names = set(mapping_dict.values())
        protein_coding = protein_coding[protein_coding["name"].isin(valid_real_names)]
        print(f"Number of protein-coding genes in {label} is: {len(protein_coding)}")

        #protein_coding.to_csv(
        #    os.path.join(gene_list_dir, f"pair_{i}_density_0.5_latent_gene_feature_top_0.1_percentile_{label}_weight_protein_coding.csv"),
        #    index=False
        #)

        txt_path = os.path.join(
            gene_list_dir,
            f"pair_{i}_dw0.5_latentgene_top_0.1_percentile_{label}_pc_genes.txt"
            )
        pd.Series(protein_coding['name'].dropna().unique()).to_csv(
            txt_path, index=False, header=False
            )

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import pickle
from scipy.stats import pearsonr
from collections import Counter
import seaborn as sns
from scipy.stats import skew
from gprofiler import GProfiler
import mygene
from collections import defaultdict
import gseapy as gp
from scipy.stats import spearmanr
from tqdm import tqdm

gene_list_dir = "/home/varnika/draelos_lab/proj_VAE/results/baseline/CGMVAE_30percent/Gene_lists_0.65/"

c1_paths = [os.path.join(gene_list_dir, f"pair_{i}_dw0.5_latentgene_top_0.1_percentile_C1_pc_genes.txt") for i in range(1, 21)]
c2_paths = [os.path.join(gene_list_dir, f"pair_{i}_dw0.5_latentgene_top_0.1_percentile_C2_pc_genes.txt") for i in range(1, 21)]

def read_gene_list(path):
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path, header=None)
    if df.empty:
        return set()
    genes = df.iloc[:, 0].dropna().astype(str)
    return set(genes)

c1_gene_sets = [read_gene_list(f) for f in c1_paths]
c2_gene_sets = [read_gene_list(f) for f in c2_paths]

for idx, gene_set in enumerate(c1_gene_sets, 1):
    print(f"Pair {idx}: C1 gene count = {len(gene_set)}")

for idx, gene_set in enumerate(c2_gene_sets, 1):
    print(f"Pair {idx}: C2 gene count = {len(gene_set)}")

c1_master_intersection = set.intersection(*c1_gene_sets) if c1_gene_sets else set()
c2_master_intersection = set.intersection(*c2_gene_sets) if c2_gene_sets else set()

with open(os.path.join(gene_list_dir, "intersection_dw0.5_C1.txt"), "w") as f:
    f.write("\n".join(sorted(c1_master_intersection)))

with open(os.path.join(gene_list_dir, "intersection_dw0.5_C2.txt"), "w") as f:
    f.write("\n".join(sorted(c2_master_intersection)))

print("\nIntersection files saved:")
print("  - intersection_dw0.5_C1.txt") # Positive effective correlation - C1
print("  - intersection_dw0.5_C2.txt") # Negative effective correlation - C2


In [ ]:
print("Number of genes in C1 intersection:", len(c1_master_intersection))
print("Number of genes in C2 intersection:", len(c2_master_intersection))

In [ ]:
c1_c2_master_intersection = c1_master_intersection & c2_master_intersection

print("Number of genes in C1 ∩ C2 (master intersection):", len(c1_c2_master_intersection))
print("Genes in C1 ∩ C2 (master intersection):")
print(sorted(c1_c2_master_intersection))

### Integrated gradients

In [ ]:
import pandas as pd
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import random
import joblib
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import psutil
from itertools import zip_longest
import time
from VAE_model_architectures import Autoencoder_CGMVAE  
device = torch.device("cpu")

In [ ]:
dim_count = 4963
num_epoch = 5000
batch_size = 511
model_type   = "C-GMVAE"
SEED         = 38
saved_epoch  = 20000

In [ ]:
rp_path = '/nfs/turbo/umms-kaczoro/yidingca/snRNA_HC_PFC_VAE/processed_count_matrix/hc_pfc_combined_rp_nor.csv'
gene_path = '/nfs/turbo/umms-kaczoro/yidingca/snRNA_HC_PFC_VAE/processed_count_matrix/hc_pfc_combined_filtered_nor.csv'
rp_pipeline_path = '/nfs/turbo/umms-kaczoro/yidingca/snRNA_HC_PFC_VAE/processed_count_matrix/random_projection.pkl'

df_genes = pd.read_csv(gene_path)
gene_counts = df_genes.iloc[:, :-11].copy()
gene_names = gene_counts.columns.tolist()

print("Gene-count matrix shape:", gene_counts.shape)
print("Number of genes:", len(gene_names))

df = pd.read_csv(rp_path, index_col=0, dtype={'Strain': str})
rp_counts = df.iloc[:, :4963].copy()
rp_feature_names = rp_counts.columns.tolist()
print("RP-count matrix shape:", rp_counts.shape)
print("Number of RP features (used):", len(rp_feature_names))

pipeline = joblib.load(rp_pipeline_path)
rp_transformer = pipeline.named_steps['random_projection']
components = rp_transformer.components_
print("RP components_ shape (n_components, n_genes):", components.shape)

# Sanity checks:
assert components.shape[0] == len(rp_feature_names), (
    f"Mismatch: components rows={components.shape[0]} "
    f"vs RP features={len(rp_feature_names)}")
assert components.shape[1] == len(gene_names), (
    f"Mismatch: components cols={components.shape[1]} "
    f"vs number of genes={len(gene_names)}")

W_df = pd.DataFrame(components,index=rp_feature_names,columns=gene_names)
gene_to_rp_importance = W_df.abs()

In [ ]:
gene_to_rp_importance

In [ ]:
def load_data(df):
    x = df.iloc[:, :dim_count].values.astype('float32')
    R_S_label = df['14-6-Bin'].values
    cfm = df['CFM_14_5_snRNA'].values

    df_layer1 = df.iloc[:, :dim_count].copy()
    df_layer1['cfm'] = cfm
    df_layer1 = df_layer1.values.astype('float32')

    return df_layer1, x, R_S_label, cfm

def numpyToTensor(x):
    return torch.from_numpy(x).to(device)

class DataBuilder(Dataset):
    def __init__(self, data):
        self.layer1, self.x, self.R_S_label, self.cfm = load_data(data)
        self.layer1   = numpyToTensor(self.layer1)
        self.x        = numpyToTensor(self.x)
        self.R_S_label = numpyToTensor(self.R_S_label)
        self.cfm      = numpyToTensor(self.cfm)
        self.len = self.x.shape[0]

    def __getitem__(self, index):
        return self.layer1[index], self.x[index], self.R_S_label[index], self.cfm[index]

    def __len__(self):
        return self.len

In [ ]:
path = '/nfs/turbo/umms-kaczoro/yidingca/snRNA_HC_PFC_VAE/processed_count_matrix/hc_pfc_combined_rp_nor.csv'
df = pd.read_csv(path, index_col=0, dtype={'Strain': str})
df['CFM_14_5_snRNA'] = (df['CFM_14_5_snRNA'] - df['CFM_14_5_snRNA'].mean()) / df['CFM_14_5_snRNA'].std()
df_rp = df.sample(frac=1, random_state=42)
test_set = DataBuilder(df_rp)
feature_names = list(df_rp.columns[:dim_count])

default_path = f"/nfs/turbo/umms-kaczoro/yidingca/snRNA_HC_PFC_VAE/trained_model_QRT_prior/{model_type}_SEED_{SEED}"
model_path   = f"{default_path}/saved_model_epoch{saved_epoch}.pth"

print("Loading model:", model_path)
model = torch.load(model_path, map_location=device) 
model.eval()

In [ ]:
import torch
import pandas as pd

def compute_latent_saliency_IG_from_testset(
    model,
    test_set,
    feature_names,
    device="cpu",
    batch_size=512,
    m_steps=50,
    baseline_type="mean",
    clip_quantile=0.99,
    eps=1e-12):
    
    if clip_quantile is not None:
        if not (0.0 < clip_quantile <= 1.0):
            raise ValueError("clip_quantile must be in (0, 1]. Use None or 1.0 to disable clipping.")

    model.to(device)
    model.eval()

    # Pull tensors
    X_all = test_set.x.to(device)                    # [N, n_features]
    y_all = test_set.R_S_label.to(device).long()     # [N]
    cfm_all = test_set.cfm.to(device)                # [N]
    N, n_features = X_all.shape

    # Determine latent_dim using a small subset
    with torch.no_grad():
        n_probe = min(10, N)
        mu_test, _, _ = model.encode(
            X_all[:n_probe],
            y_all[:n_probe],
            cfm_all[:n_probe],
        )
        latent_dim = mu_test.shape[1]

    # Baseline vector
    if baseline_type == "mean":
        baseline_vec = X_all.mean(dim=0, keepdim=True)  # [1, n_features]
    elif baseline_type == "zero":
        baseline_vec = torch.zeros(1, n_features, device=device)
    else:
        raise ValueError(f"Unknown baseline_type: {baseline_type}")

    # Accumulate SUM of winsorized |IG| over ALL cells (not per-batch means)
    ig_sum_abs = torch.zeros(latent_dim, n_features, device=device)
    total_cells = 0

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)

        x_batch = X_all[start:end]          # [B, n_features]
        y_batch = y_all[start:end]          # [B]
        cfm_batch = cfm_all[start:end]      # [B]
        B = x_batch.shape[0]

        baseline_batch = baseline_vec.expand(B, -1)  # [B, n_features]
        diff = x_batch - baseline_batch              # [B, n_features]

        # For each latent dimension, compute IG over the path
        for k in range(latent_dim):
            grads_accum = torch.zeros_like(x_batch, device=device)

            for step in range(1, m_steps + 1):
                alpha = float(step) / m_steps
                x_step = baseline_batch + alpha * diff
                x_step.requires_grad_(True)

                mu_step, _, _ = model.encode(x_step, y_batch, cfm_batch)
                mu_k = mu_step[:, k].sum()  # scalar

                model.zero_grad(set_to_none=True)
                if x_step.grad is not None:
                    x_step.grad.zero_()

                mu_k.backward()
                grads_accum += x_step.grad.detach()  # [B, n_features]

            avg_grads = grads_accum / float(m_steps)            # [B, n_features]
            ig_batch = diff * avg_grads                         # [B, n_features]
            ig_abs = ig_batch.abs()                             # [B, n_features]

            # Winsorize (clip) per LV k and per feature (gene) across cells in this batch
            if clip_quantile is not None and clip_quantile < 1.0:
                # cap per feature: [1, n_features]
                # torch.quantile handles float tensors; ensure numerical stability
                cap = torch.quantile(ig_abs, clip_quantile, dim=0, keepdim=True)
                cap = torch.clamp(cap, min=eps)
                ig_abs = torch.minimum(ig_abs, cap)

            # Sum over cells (weighted correctly by batch size)
            ig_sum_abs[k] += ig_abs.sum(dim=0)                  # [n_features]

        total_cells += B

    # Mean(|IG|) across ALL cells
    ig_avg = (ig_sum_abs / max(total_cells, 1)).detach().cpu().numpy()

    latent_names = [f"LV{i+1}" for i in range(latent_dim)]
    saliency_df = pd.DataFrame(ig_avg, index=latent_names, columns=feature_names)
    return saliency_df

In [ ]:
import os
out_dir = r'/nfs/turbo/umms-kaczoro/yidingca/snRNA_HC_PFC_VAE/interpretability_outputs'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
saliency_df_ig = compute_latent_saliency_IG_from_testset(
    model=model,
    test_set=test_set,
    feature_names=feature_names,
    device="cpu",        
    batch_size=batch_size,     
    m_steps=50,          
    baseline_type="zero")

In [ ]:
saliency_df_ig

In [ ]:
# Align RP dimensions
saliency_df_ig.columns = saliency_df_ig.columns.astype(str)
gene_to_rp_importance.index = gene_to_rp_importance.index.astype(str)
common_rps = saliency_df_ig.columns.intersection(gene_to_rp_importance.index)

A = saliency_df_ig.loc[:, common_rps].astype(float)
B = gene_to_rp_importance.loc[common_rps, :].astype(float)

# Chain rule
gene_to_lv_importance = A @ B    # (LV, gene)

# Remove pandas index
# gene_to_lv_importance.reset_index(drop=True, inplace=True)

In [ ]:
gene_to_lv_importance

In [ ]:
gene_names = pd.read_csv('/nfs/turbo/umms-kaczoro/yidingca/snRNA_HC_PFC_VAE/raw_count_matrix/hc_pfc_overlap_genes.csv')
gene_to_lv_importance.columns = gene_names['gene'].values

In [ ]:
gene_to_lv_importance

In [ ]:
gene_to_lv_importance.to_csv(os.path.join(out_dir, "gene_to_LV_importance.csv"), index=True)

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, Optional


def _elbow_index_desc_chord(scores_desc: np.ndarray, min_genes: int = 5, max_search: Optional[int] = None) -> int:
    y_full = np.asarray(scores_desc, dtype=float)
    n_full = y_full.size
    if n_full == 0:
        return -1

    y = y_full if max_search is None else y_full[: min(n_full, int(max_search))]
    n = y.size
    if n <= max(2, min_genes):
        return n - 1

    x = np.arange(n, dtype=float)

    y_min, y_max = float(y.min()), float(y.max())
    if np.isclose(y_max, y_min):
        return n - 1
    y_norm = (y - y_min) / (y_max - y_min)

    x1, y1 = x[0], y_norm[0]
    x2, y2 = x[-1], y_norm[-1]

    denom = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)
    if np.isclose(denom, 0.0):
        return n - 1

    dist = np.abs((y2 - y1) * x - (x2 - x1) * y_norm + (x2 * y1 - y2 * x1)) / denom

    start = max(1, min_genes - 1)
    end = n - 2
    if start > end:
        return min_genes - 1

    return int(np.argmax(dist[start:end + 1]) + start)


def get_elbow_genes_exact_like_plot(
    gene_to_lv_importance: pd.DataFrame,
    gamma: float = 10.0,
    min_genes: int = 5,
    include_elbow: bool = True,
    dropna: bool = True,
    max_search: Optional[int] = None,
) -> Dict[str, pd.DataFrame]:
    """
    EXACTLY matches your plotting transformation per LV:
        mu = mean(lv_sorted)
        lv_power = mu * (lv_sorted/mu)**gamma   (if mu!=0 else lv_sorted)

    Elbow is computed on lv_power (not original scores).
    Returned genes/scores are ORIGINAL (untransformed) importance, descending.
    """
    out: Dict[str, pd.DataFrame] = {}

    for lv in gene_to_lv_importance.index:
        lv_scores = gene_to_lv_importance.loc[lv]
        if dropna:
            lv_scores = lv_scores.dropna()

        lv_scores = pd.to_numeric(lv_scores, errors="coerce").dropna()
        if lv_scores.empty:
            out[lv] = pd.DataFrame(columns=["gene", "importance"])
            continue

        lv_sorted = lv_scores.sort_values(ascending=False)

        mu = lv_sorted.mean()
        if mu != 0:
            lv_power = mu * (lv_sorted / mu) ** gamma
        else:
            lv_power = lv_sorted.copy()

        elbow_idx = _elbow_index_desc_chord(
            lv_power.values,   # elbow uses transformed values
            min_genes=min_genes,
            max_search=max_search,
        )

        cut = elbow_idx + 1 if include_elbow else elbow_idx
        cut = max(cut, min_genes)
        cut = min(cut, len(lv_sorted))

        # Return ORIGINAL scores (lv_sorted), not lv_power
        out[str(lv)] = pd.DataFrame({
            "gene": lv_sorted.index[:cut].to_numpy(),
            "importance": lv_sorted.values[:cut],
            "importance_power": lv_power.values[:cut],  # include to verify exact match
            "rank_1based": np.arange(1, cut + 1)
        })

    return out


def elbow_dict_to_long_df(elbow_genes: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    frames = []
    for lv, df in elbow_genes.items():
        if df is None or df.empty:
            continue
        tmp = df.copy()
        tmp.insert(0, "LV", lv)
        frames.append(tmp)
    if not frames:
        return pd.DataFrame(columns=["LV", "gene", "importance", "importance_power", "rank_1based"])
    return pd.concat(frames, ignore_index=True)

In [ ]:
elbow_genes = get_elbow_genes_exact_like_plot(gene_to_lv_importance, gamma=10, min_genes=10)
elbow_genes

In [ ]:
import os
for lv, df in elbow_genes.items():
    out_path = os.path.join(out_dir, f"elbow_genes_{lv}.csv")
    df.to_csv(out_path, index=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

gamma =10
pad = 0.05  # 5% x-axis padding (try 0.1 for bigger gap)

n_lv = gene_to_lv_importance.shape[0]

ncols = 3
nrows = math.ceil(n_lv / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4*nrows))
axes = axes.flatten()

for i in range(n_lv):
    lv_scores = pd.to_numeric(gene_to_lv_importance.iloc[i, :], errors="coerce").dropna()
    lv_sorted = lv_scores.sort_values(ascending=False)

    mu = lv_sorted.mean()
    if mu != 0:
        lv_power = mu * (lv_sorted / mu) ** gamma
    else:
        lv_power = lv_sorted.copy()

    x = np.arange(1, len(lv_power) + 1)

    axes[i].plot(x, lv_power.values)
    axes[i].set_title(f"LV{i+1}")
    axes[i].set_xlabel("Gene Importance rank")
    axes[i].set_ylabel("IG importance score")

    # manual x padding (because set_xlim disables margins)
    xmin, xmax = 1, len(lv_power)
    xr = max(1, xmax - xmin)  # avoid zero division if only 1 point
    axes[i].set_xlim(xmin - pad * xr, xmax + pad * xr)

    # keep ticks starting at 1 (optional; remove if you want auto ticks)
    # axes[i].set_xticks([1, xmax])

# remove empty subplots
for j in range(n_lv, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()